# ============================================================
# SECTION 1 — DATA LOADING AND VALIDATION
# Multi-Resolution Semantic Abstraction over an Evolving TKH
# ============================================================

In [ ]:
from pathlib import Path
from collections import Counter
import json
import random

import numpy as np
import pandas as pd

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

print(f"Seed: {SEED}")

In [ ]:
DATA_DIR = Path("data")

TKH_PATH       = DATA_DIR / "tkh_collection10.json"
QUESTIONS_PATH = DATA_DIR / "questions.csv"
GOLD_PATH      = DATA_DIR / "ground_truth.json"
ARTICLES_PATH  = DATA_DIR / "collection10_articles.csv"

FILES = {
    "TKH": TKH_PATH,
    "Questions": QUESTIONS_PATH,
    "Ground truth": GOLD_PATH,
    "Articles": ARTICLES_PATH,
}

for name, path in FILES.items():
    print(f"{name:15s}: {path} -> exists={path.exists()}")

In [ ]:
# ---------- TKH ----------
with open(TKH_PATH, "r", encoding="utf-8") as f:
    tkh = json.load(f)

# ---------- Questions ----------
questions = pd.read_csv(
    QUESTIONS_PATH,
    sep=";",
    dtype={
        "question_id": "string",
        "question": "string",
        "type": "string",
    },
)

# ---------- Ground truth ----------
with open(GOLD_PATH, "r", encoding="utf-8") as f:
    ground_truth = json.load(f)

# ---------- Article metadata ----------
articles = pd.read_csv(
    ARTICLES_PATH,
    dtype={
        "id": "Int64",
        "year": "Int64",
        "arxiv_id": "string",
        "status": "string",
        "title": "string",
    },
)

print("All files loaded successfully.")

In [ ]:
print("TKH top-level keys:")
print(list(tkh.keys()))

print("\nTKH metadata:")
print(f"Collection ID : {tkh['meta']['collection_id']}")
print(f"Articles      : {tkh['meta']['articles']}")
print(f"Nodes         : {tkh['meta']['nodes']}")
print(f"Hyperedges    : {tkh['meta']['hyperedges']}")

print("\nBenchmark:")
print(f"Questions     : {len(questions)}")
print(f"Gold entries  : {len(ground_truth)}")
print(f"Article rows  : {len(articles)}")

The assessment description mentions 17 benchmark questions, but the supplied benchmark actually contains 18 questions, and the ground truth contains matching entries for all 18.

In [ ]:
nodes = tkh["nodes"]
hyperedges = tkh["hyperedges"]

node_ids = [node["id"] for node in nodes]
edge_ids = [edge["id"] for edge in hyperedges]

node_id_set = set(node_ids)
edge_id_set = set(edge_ids)

print(f"Loaded {len(nodes):,} nodes")
print(f"Loaded {len(hyperedges):,} hyperedges")

### Validate required schema

In [ ]:
REQUIRED_NODE_FIELDS = {
    "id",
    "type",
    "surface_form",
    "year",
    "origin_year",
    "first_seen_year",
    "last_seen_year",
    "provenance",
}

REQUIRED_EDGE_FIELDS = {
    "id",
    "relation_type",
    "members",
    "year",
    "provenance",
}

bad_nodes = [
    node["id"]
    for node in nodes
    if not REQUIRED_NODE_FIELDS.issubset(node.keys())
]

bad_edges = [
    edge["id"]
    for edge in hyperedges
    if not REQUIRED_EDGE_FIELDS.issubset(edge.keys())
]

print("Nodes with missing required fields :", len(bad_nodes))
print("Edges with missing required fields :", len(bad_edges))

assert len(bad_nodes) == 0, f"Malformed nodes found: {bad_nodes[:10]}"
assert len(bad_edges) == 0, f"Malformed hyperedges found: {bad_edges[:10]}"

print("Schema validation passed.")

### Validate unique IDs

In [ ]:
duplicate_node_ids = len(node_ids) - len(node_id_set)
duplicate_edge_ids = len(edge_ids) - len(edge_id_set)

print("Duplicate node IDs :", duplicate_node_ids)
print("Duplicate edge IDs :", duplicate_edge_ids)

assert duplicate_node_ids == 0
assert duplicate_edge_ids == 0

print("ID uniqueness validation passed.")

### Validate hyperedge membership

In [ ]:
dangling_members = []

for edge in hyperedges:
    missing = [
        member
        for member in edge["members"]
        if member not in node_id_set
    ]

    if missing:
        dangling_members.append({
            "edge_id": edge["id"],
            "missing_members": missing,
        })

print("Hyperedges with dangling node references:", len(dangling_members))

assert len(dangling_members) == 0, dangling_members[:5]

print("Hyperedge membership validation passed.")

### Validate hyperedge arity

In [ ]:
arities = np.array([
    len(edge["members"])
    for edge in hyperedges
])

print(f"Minimum arity : {arities.min()}")
print(f"Maximum arity : {arities.max()}")
print(f"Mean arity    : {arities.mean():.2f}")
print(f"Median arity  : {np.median(arities):.1f}")

multiway_fraction = np.mean(arities > 2)

print(
    f"Hyperedges with arity > 2: "
    f"{multiway_fraction:.2%}"
)

assert arities.min() >= 2

It proves why simply converting this problem into a conventional pairwise graph would be questionable: roughly four out of five relations genuinely contain more than two participants.

### Validate surface forms

In [ ]:
empty_surface_forms = [
    node["id"]
    for node in nodes
    if not str(node.get("surface_form", "")).strip()
]

print("Nodes without usable surface_form:", len(empty_surface_forms))

assert len(empty_surface_forms) == 0

print("All nodes contain usable semantic text.")

That's very useful later because every node can receive a semantic representation.

### Validate node and relation type counts

In [ ]:
actual_node_types = Counter(
    node["type"]
    for node in nodes
)

actual_relation_types = Counter(
    edge["relation_type"]
    for edge in hyperedges
)

print("NODE TYPES")
for node_type, count in sorted(actual_node_types.items()):
    print(f"{node_type:15s}: {count:4d}")

print("\nRELATION TYPES")
for relation, count in sorted(actual_relation_types.items()):
    print(f"{relation:25s}: {count:4d}")

assert dict(actual_node_types) == tkh["meta"]["node_types"]
assert dict(actual_relation_types) == tkh["meta"]["relation_types"]

print("\nMetadata counts exactly match actual JSON contents.")

### Check temporal fields carefully

In [ ]:
TEMPORAL_FIELDS = [
    "year",
    "origin_year",
    "first_seen_year",
    "last_seen_year",
]

for field in TEMPORAL_FIELDS:

    values = [
        node[field]
        for node in nodes
        if node.get(field) is not None
    ]

    missing = len(nodes) - len(values)

    print(
        f"{field:16s} "
        f"min={min(values):4d} "
        f"max={max(values):4d} "
        f"available={len(values):4d} "
        f"missing={missing:4d}"
    )

This tells us something very important:

origin_year is unavailable for most nodes:

$$ 4967 / 5798 \approx 85.7\% $$

so it cannot serve as the sole temporal visibility criterion.

### Validate temporal consistency

In [ ]:
invalid_seen_intervals = []

for node in nodes:

    first_seen = node.get("first_seen_year")
    last_seen = node.get("last_seen_year")

    if (
        first_seen is not None
        and last_seen is not None
        and first_seen > last_seen
    ):
        invalid_seen_intervals.append(node["id"])

print(
    "Nodes where first_seen_year > last_seen_year:",
    len(invalid_seen_intervals)
)

assert len(invalid_seen_intervals) == 0

print("Temporal interval validation passed.")

### Quantify the crucial year vs first_seen_year distinction

In [ ]:
year_before_first_seen = []
year_after_first_seen = []
year_equal_first_seen = []

for node in nodes:

    year = node.get("year")
    first_seen = node.get("first_seen_year")

    if year is None or first_seen is None:
        continue

    if year < first_seen:
        year_before_first_seen.append(node["id"])

    elif year > first_seen:
        year_after_first_seen.append(node["id"])

    else:
        year_equal_first_seen.append(node["id"])

print(
    "year < first_seen_year :",
    len(year_before_first_seen)
)

print(
    "year = first_seen_year :",
    len(year_equal_first_seen)
)

print(
    "year > first_seen_year :",
    len(year_after_first_seen)
)

The 716 cases are extremely relevant.

Why?

Because year can represent when a concept or method originated, while first_seen_year represents when it became visible in this corpus.

Therefore:

node["year"] <= cutoff

does not necessarily mean:

this information was available to the TKH at that cutoff.

We'll resolve this formally in Section 2.

### Inspect examples of that temporal distinction

In [ ]:
examples = []

for node in nodes:

    year = node["year"]
    first_seen = node["first_seen_year"]

    if year < first_seen:
        examples.append({
            "id": node["id"],
            "type": node["type"],
            "surface_form": node["surface_form"],
            "year": year,
            "origin_year": node["origin_year"],
            "first_seen_year": first_seen,
        })

examples_df = pd.DataFrame(examples)

examples_df.head(15)

### Cross-check the 52 article nodes against collection10_articles.csv

In [ ]:
article_nodes = [
    node
    for node in nodes
    if node["type"] == "article"
]

print("Article nodes in TKH:", len(article_nodes))
print("Rows in article CSV  :", len(articles))

assert len(article_nodes) == 52
assert len(articles) == 52

In [ ]:
article_id_to_node = {}

for node in article_nodes:

    provenance_articles = (
        node.get("provenance", {})
        .get("articles", [])
    )

    for article_id in provenance_articles:
        article_id_to_node[int(article_id)] = node


csv_article_ids = set(
    articles["id"]
    .dropna()
    .astype(int)
)

tkh_article_ids = set(article_id_to_node.keys())

print(
    "Article IDs only in CSV:",
    csv_article_ids - tkh_article_ids
)

print(
    "Article IDs only in TKH:",
    tkh_article_ids - csv_article_ids
)

assert csv_article_ids == tkh_article_ids

print("Article IDs match exactly.")

### Cross-check article titles and publication years

files are completely aligned.

In [ ]:
article_mismatches = []

for _, row in articles.iterrows():

    article_id = int(row["id"])

    node = article_id_to_node[article_id]

    csv_title = str(row["title"]).strip()
    tkh_title = str(node["surface_form"]).strip()

    csv_year = int(row["year"])
    tkh_year = int(node["year"])

    if (
        csv_title != tkh_title
        or csv_year != tkh_year
    ):
        article_mismatches.append({
            "article_id": article_id,
            "csv_title": csv_title,
            "tkh_title": tkh_title,
            "csv_year": csv_year,
            "tkh_year": tkh_year,
        })


print(
    "Article title/year mismatches:",
    len(article_mismatches)
)

assert len(article_mismatches) == 0

print("Article metadata matches TKH exactly.")

### Validate questions against ground truth

In [ ]:
question_ids = questions["question_id"].tolist()

print("Questions:", len(question_ids))
print("Unique question IDs:", len(set(question_ids)))
print("Ground-truth entries:", len(ground_truth))

assert len(question_ids) == len(set(question_ids))

assert set(question_ids) == set(ground_truth.keys())

print("Question IDs and ground-truth IDs match exactly.")

### Validate benchmark question types

In [ ]:
print("Question types from CSV:")
print(questions["type"].value_counts())

gold_types = Counter(
    item["type"]
    for item in ground_truth.values()
)

print("\nQuestion types from ground truth:")
print(gold_types)

In [ ]:
benchmark_errors = []

for qid, item in ground_truth.items():

    qtype = item.get("type")

    if qtype == "A":

        if "expected_methods" not in item:
            benchmark_errors.append(
                f"{qid}: Type A missing expected_methods"
            )

    elif qtype == "B":

        if "required_claims" not in item:
            benchmark_errors.append(
                f"{qid}: Type B missing required_claims"
            )

    else:

        benchmark_errors.append(
            f"{qid}: Unknown type {qtype}"
        )


print(
    "Benchmark schema errors:",
    len(benchmark_errors)
)

assert len(benchmark_errors) == 0

print("Ground-truth benchmark schema is valid.")

### Final Section 1 validation report

In [ ]:
section1_report = {
    "collection_id": tkh["meta"]["collection_id"],

    "num_nodes": len(nodes),
    "num_hyperedges": len(hyperedges),

    "num_articles": len(articles),

    "num_questions": len(questions),
    "num_type_A_questions":
        int((questions["type"] == "A").sum()),

    "num_type_B_questions":
        int((questions["type"] == "B").sum()),

    "duplicate_node_ids": duplicate_node_ids,
    "duplicate_edge_ids": duplicate_edge_ids,

    "dangling_hyperedge_members":
        len(dangling_members),

    "empty_surface_forms":
        len(empty_surface_forms),

    "min_hyperedge_arity":
        int(arities.min()),

    "max_hyperedge_arity":
        int(arities.max()),

    "fraction_hyperedges_arity_gt_2":
        float(np.mean(arities > 2)),

    "nodes_year_before_first_seen":
        len(year_before_first_seen),

    "article_metadata_mismatches":
        len(article_mismatches),

    "benchmark_schema_errors":
        len(benchmark_errors),
}

section1_report

## Section 1 has established



*   TKH has 5,798 nodes and 1,429 genuine hyperedges.
*   Hyperedge arity ranges from 2 to 65.
*   TKH has 5,798 nodes and 1,429 genuine hyperedges.



*   Approximately 80.5% of hyperedges have more than two endpoints.
*   All hyperedge members reference valid nodes.
*   All nodes have semantic text.
*   No duplicate node or hyperedge IDs exist.
*   All 52 article metadata records agree with TKH article nodes.
*   There are 18 benchmark questions, not 17 in the actual supplied files.
*   Q1–Q14 are Type A and Q15–Q18 are Type B.
*   year and first_seen_year are materially different: 716 nodes have year < first_seen_year.
*   Therefore temporal visibility cannot safely be defined using node.year alone.

In [89]:
!find /content/drive/MyDrive/Apply/Germany/ConstructorLabs -maxdepth 2 -name "*.ipynb" -print

/content/drive/MyDrive/Apply/Germany/ConstructorLabs/01_load_and_validate_data.ipynb
/content/drive/MyDrive/Apply/Germany/ConstructorLabs/Copy of tkh_hierarchy_main.ipynb


In [90]:
from pathlib import Path

CLEAN = Path("/content/drive/MyDrive/Apply/Germany/ConstructorLabs/01_load_and_validate_data.ipynb")
REPO_FILE = Path(
    "/content/tkh-hierarchy-project/"
    "notebooks/01_load_and_validate_data.ipynb"
)

print("Clean file exists:", CLEAN.exists())
print("Repo file exists :", REPO_FILE.exists())

Clean file exists: True
Repo file exists : True


In [91]:
import re

clean_text = CLEAN.read_text(encoding="utf-8")

patterns = {
    "classic PAT": r"ghp_[A-Za-z0-9]{20,}",
    "fine-grained PAT": r"github_pat_[A-Za-z0-9_]{20,}",
    "authenticated GitHub URL":
        r"https://[^/\s\"]+:[^@\s\"]+@github\.com",
}

for name, pattern in patterns.items():
    print(
        f"{name}: "
        f"{'FOUND' if re.search(pattern, clean_text) else 'clean'}"
    )

classic PAT: clean
fine-grained PAT: clean
authenticated GitHub URL: clean


In [92]:
import shutil

shutil.copy2(CLEAN, REPO_FILE)

print("Clean notebook copied into repository.")

Clean notebook copied into repository.


In [93]:
repo_text = REPO_FILE.read_text(encoding="utf-8")

for name, pattern in patterns.items():
    print(
        f"{name}: "
        f"{'FOUND' if re.search(pattern, repo_text) else 'clean'}"
    )

classic PAT: clean
fine-grained PAT: clean
authenticated GitHub URL: clean


In [94]:
import hashlib

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

print("Clean SHA :", sha256(CLEAN))
print("Repo SHA  :", sha256(REPO_FILE))

assert sha256(CLEAN) == sha256(REPO_FILE)

print("✓ Files are identical.")

Clean SHA : 7f5ef110af490a5e355fde0d60f255c89170c824531c3b1cc11a8fe8fd0e0d19
Repo SHA  : 22dda71d3273421ef149a5b1706696af2f391bfd72561b083c1565e782dd6437


AssertionError: 